# CRYPTEX DataRobot Workbench Notebook

This notebook is a self-contained, reproducible workflow for user-uploaded Elliptic CSV files.
It validates headers and labels, keeps unknown labels out of training/evaluation ground truth, runs chronological splitting, tunes a decision threshold on validation only, evaluates on hold-out test data, and exports report-ready tables.


## Dependencies
Install in a notebook cell if needed:
```python
%pip install -q pandas numpy scikit-learn lightgbm matplotlib seaborn
```


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score, roc_auc_score, confusion_matrix
from sklearn.ensemble import HistGradientBoostingClassifier

try:
    import lightgbm as lgb
except Exception:
    lgb = None


In [ ]:
# ---- User configuration ----
CLASSES_PATH = Path('/workspace/data/elliptic_txs_classes.csv')  # Upload path in Workbench
FEATURES_PATH = Path('/workspace/data/elliptic_txs_features.csv')  # Upload path in Workbench
OUTPUT_DIR = Path('/workspace/outputs/cryptex_notebook')
TRAIN_END_STEP = 34
TUNE_END_STEP = 39
TEST_END_STEP = 49
THRESHOLDS = np.linspace(0.05, 0.95, 91)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
def map_labels(series: pd.Series) -> pd.Series:
    norm = series.astype(str).str.strip().str.lower()
    mapped = norm.map({
        '1': 1, 'illicit': 1,
        '2': 0, '0': 0, 'licit': 0,
        'unknown': np.nan, 'nan': np.nan, 'none': np.nan, '': np.nan
    })
    numeric = pd.to_numeric(series, errors='coerce')
    mapped = mapped.where(~mapped.isna(), numeric)
    return mapped.where(mapped.isin([0, 1]), np.nan)

def validate_headers(classes_df: pd.DataFrame, features_df: pd.DataFrame):
    classes_required = {'txId', 'class'}
    features_required = {'txId', 'time_step'}
    missing_classes = classes_required - set(classes_df.columns)
    missing_features = features_required - set(features_df.columns)
    if missing_classes:
        raise ValueError(f'Missing required class headers: {sorted(missing_classes)}')
    if missing_features:
        raise ValueError(f'Missing required feature headers: {sorted(missing_features)}')
    feature_cols = [c for c in features_df.columns if c.startswith('feat_')]
    if not feature_cols:
        raise ValueError('No feature columns found. Expected one or more feat_* columns.')
    return feature_cols

def load_elliptic(classes_path: Path, features_path: Path):
    classes_df = pd.read_csv(classes_path)
    features_df = pd.read_csv(features_path)
    feature_cols = validate_headers(classes_df, features_df)

    classes_df = classes_df.copy()
    classes_df['label'] = map_labels(classes_df['class'])
    classes_df['is_known_label'] = classes_df['label'].isin([0, 1])

    merged = features_df.merge(classes_df[['txId', 'label', 'is_known_label']], on='txId', how='left')
    merged['time_step'] = pd.to_numeric(merged['time_step'], errors='coerce')
    if merged['time_step'].isna().any():
        raise ValueError('time_step contains non-numeric values')
    merged['time_step'] = merged['time_step'].astype(int)

    known = merged[merged['is_known_label']].copy()
    known['label'] = known['label'].astype(int)
    return merged, known, feature_cols

def temporal_split(df_known: pd.DataFrame):
    train = df_known[df_known['time_step'] <= TRAIN_END_STEP].copy()
    tune = df_known[(df_known['time_step'] > TRAIN_END_STEP) & (df_known['time_step'] <= TUNE_END_STEP)].copy()
    test = df_known[(df_known['time_step'] > TUNE_END_STEP) & (df_known['time_step'] <= TEST_END_STEP)].copy()
    if not train.empty and not tune.empty and train['time_step'].max() >= tune['time_step'].min():
        raise ValueError('Train/tune overlap detected')
    if not tune.empty and not test.empty and tune['time_step'].max() >= test['time_step'].min():
        raise ValueError('Tune/test overlap detected')
    return train, tune, test

def choose_threshold(y_true, y_prob, thresholds):
    best = {'threshold': 0.5, 'f1': -1, 'precision': -1, 'recall': -1}
    for t in thresholds:
        pred = (y_prob >= t).astype(int)
        p = precision_score(y_true, pred, zero_division=0)
        r = recall_score(y_true, pred, zero_division=0)
        f1 = f1_score(y_true, pred, zero_division=0)
        if (f1 > best['f1']) or (np.isclose(f1, best['f1']) and p > best['precision']):
            best = {'threshold': float(t), 'f1': float(f1), 'precision': float(p), 'recall': float(r)}
    return best

def evaluate(y_true, y_prob, threshold):
    pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {
        'threshold': float(threshold),
        'precision': precision_score(y_true, pred, zero_division=0),
        'recall': recall_score(y_true, pred, zero_division=0),
        'f1': f1_score(y_true, pred, zero_division=0),
        'auprc': average_precision_score(y_true, y_prob) if np.sum(y_true == 1) > 0 else 0.0,
        'roc_auc': roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else 0.0,
        'tp': int(tp), 'fp': int(fp), 'fn': int(fn), 'tn': int(tn)
    }


In [ ]:
all_rows, known_rows, feature_cols = load_elliptic(CLASSES_PATH, FEATURES_PATH)
train_df, tune_df, test_df = temporal_split(known_rows)

print('All transactions:', len(all_rows))
print('Known-label transactions:', len(known_rows))
print('Unknown-label transactions:', len(all_rows) - len(known_rows))
print(f'Train={len(train_df)} Tune={len(tune_df)} Test={len(test_df)}')
known_rows[['time_step','label']].groupby('time_step')['label'].agg(['count','mean']).head()


In [ ]:
X_train, y_train = train_df[feature_cols], train_df['label'].values
X_tune, y_tune = tune_df[feature_cols], tune_df['label'].values
X_test, y_test = test_df[feature_cols], test_df['label'].values

if lgb is not None:
    model = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05, max_depth=6, class_weight='balanced', random_state=42, verbose=-1)
else:
    model = HistGradientBoostingClassifier(max_depth=6, random_state=42)

model.fit(X_train, y_train)
tune_prob = model.predict_proba(X_tune)[:, 1]
threshold_info = choose_threshold(y_tune, tune_prob, THRESHOLDS)
threshold = threshold_info['threshold']
print('Threshold selected from tune only:', threshold_info)


In [ ]:
test_prob = model.predict_proba(X_test)[:, 1]
overall_metrics = evaluate(y_test, test_prob, threshold)
metrics_df = pd.DataFrame([overall_metrics])
metrics_df


In [ ]:
# Per-time-step holdout evaluation for report tables
rows = []
for step, step_df in test_df.groupby('time_step'):
    y_step = step_df['label'].values
    p_step = model.predict_proba(step_df[feature_cols])[:, 1]
    m = evaluate(y_step, p_step, threshold)
    m['time_step'] = int(step)
    m['num_illicit'] = int(np.sum(y_step == 1))
    m['total_tx'] = int(len(step_df))
    rows.append(m)

step_metrics_df = pd.DataFrame(rows).sort_values('time_step')
step_metrics_df.head()


In [ ]:
# Export report-ready outputs
metrics_df.to_csv(OUTPUT_DIR / 'overall_holdout_metrics.csv', index=False)
step_metrics_df.to_csv(OUTPUT_DIR / 'per_step_holdout_metrics.csv', index=False)

summary_table = step_metrics_df[['time_step','auprc','precision','recall','f1','tp','fp','fn','tn','num_illicit','total_tx']]
summary_table.to_csv(OUTPUT_DIR / 'table_holdout_step_metrics.csv', index=False)
summary_table.to_markdown(OUTPUT_DIR / 'table_holdout_step_metrics.md', index=False)

print('Saved outputs to', OUTPUT_DIR)


## Notes
- Unknown labels are excluded from model fitting and metrics.
- Threshold is selected on tune data only, not on the final test set.
- Re-run after changing paths to newly uploaded files.
